# Spotify Dataset Visualization

This notebook splits the visualization script into separate cells for each region:
1. Imports and styling
2. Data loading
3. Data cleaning and feature setup
4. Correlation heatmap
5. Feature distribution plots
6. Top genres countplot
7. Pairplot analysis

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

sns.set_theme(style="whitegrid")
plt.rcParams["figure.facecolor"] = "whitesmoke"

In [ ]:
dataset_path = os.path.join("..", "data", "raw", "spotify_tracks.csv")
output_dir = os.path.join("..", "plots")
os.makedirs(output_dir, exist_ok=True)

if not os.path.exists(dataset_path):
    raise FileNotFoundError(f"Spotify tracks dataset not found at '{dataset_path}'.")

df = pd.read_csv(dataset_path)
print(f"Loaded {df.shape[0]} tracks from the raw dataset.")

In [ ]:
audio_features = ["danceability", "energy", "acousticness", "tempo", "valence"]

df_clean = df.dropna(subset=audio_features + ["track_genre"])
print(f"Cleaned dataset shape: {df_clean.shape}")

## Correlation Heatmap

Visualize the Pearson correlation between key audio features.

In [ ]:
corr_matrix = df_clean[audio_features].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(
    corr_matrix,
    annot=True,
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    fmt=".2f",
    linewidths=0.5,
    cbar_kws={"label": "Correlation Coefficient"}
)
plt.title("Correlation Heatmap of Key Audio Features", fontsize=14, fontweight="bold", pad=15)
plt.tight_layout()
plt.show()

## Audio Feature Distributions

Plot the distribution of each selected audio feature.

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 12))
axes = axes.flatten()

for idx, feature in enumerate(audio_features):
    sns.histplot(
        df_clean[feature],
        kde=True,
        ax=axes[idx],
        color="teal",
        bins=30,
        edgecolor="white"
    )
    axes[idx].set_title(f"Distribution of {feature.capitalize()}", fontsize=12, fontweight="semibold")
    axes[idx].set_xlabel(feature.capitalize())
    axes[idx].set_ylabel("Count")

fig.delaxes(axes[5])
plt.suptitle("Distribution Analysis of Key Musical Attributes", fontsize=16, fontweight="bold", y=0.98)
plt.tight_layout()
plt.show()

## Top Genres Countplot

Show the top 10 genres by track count, if the genre column exists.

In [ ]:
if "track_genre" in df_clean.columns:
    top_genres = df_clean["track_genre"].value_counts().head(10)
    plt.figure(figsize=(10, 6))
    sns.barplot(
        x=top_genres.values,
        y=top_genres.index,
        palette="viridis",
        edgecolor="white"
    )
    plt.title("Top 10 Music Genres in Spotify Dataset", fontsize=14, fontweight="bold", pad=15)
    plt.xlabel("Number of Tracks")
    plt.ylabel("Genre")
    plt.tight_layout()
    plt.show()
else:
    print("No 'track_genre' column found; skipping genre countplot.")

## Pairplot of Audio Features

Create a pairplot to explore relationships between the selected audio features.

In [ ]:
df_sample = df_clean[audio_features + ["track_genre"]].sample(min(1500, len(df_clean)), random_state=42)
pair_grid = sns.pairplot(
    df_sample,
    vars=audio_features,
    kind="scatter",
    diag_kind="kde",
    plot_kws={"alpha": 0.5, "color": "darkorchid", "s": 15},
    diag_kws={"color": "darkorchid", "fill": True}
)
pair_grid.fig.suptitle("Pairplot Matrix of Key Audio Attributes", fontsize=16, fontweight="bold", y=1.02)
plt.show()